# nova_ai 快速开始

本 Notebook 演示如何用最少的代码调用 `nova_agent` 底层的 `nova_ai` 流式 API。

运行前请确保已设置对应厂商的 API Key，例如 `VOLCENGINE_API_KEY`。


In [2]:
import os

# 请替换为你的真实 API Key
os.environ["VOLCENGINE_API_KEY"] = "3b631f71-6bd6-464a-9abc-b0e8d19f25d7"
# 如果使用其他厂商，可设置 OPENAI_API_KEY、ANTHROPIC_API_KEY 等


## 发起一次流式对话

`stream_simple` 返回一个异步事件流 `AssistantMessageEventStream`，可通过 `async for` 消费事件，最后通过 `await stream.result()` 获取完整的 `AssistantMessage`。


In [3]:
from nova_ai import get_model, UserMessage, Context, stream_simple

model = get_model("volcengine", "deepseek-v3-2-251201")

context = Context(
    system_prompt="你是一个有帮助的助手。",
    messages=[UserMessage(role="user", content="你好，请用一句话介绍自己。")],
)

stream = stream_simple(model, context)

async for event in stream:
    if event.type == "text_delta":
        print(event.delta, end="", flush=True)
    elif event.type == "thinking_delta":
        print(f"\n[思考] {event.delta}", end="")
    elif event.type == "done":
        print("\n\n--- 完成 ---")

final_message = await stream.result()
print("\n完整回复:", final_message.content[0].text if final_message.content else "")
print("停止原因:", final_message.stop_reason)


你好，我是一个由深度求索公司创造的AI助手，致力于为你提供准确、有用的信息和帮助。

--- 完成 ---

完整回复: 你好，我是一个由深度求索公司创造的AI助手，致力于为你提供准确、有用的信息和帮助。
停止原因: stop
